In [ ]:
# ===========================================================
# FULL SAFE NOTEBOOK – Data Preparation + Feature Engineering
# ===========================================================

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ===========================================================
# 0. Define file paths
# ===========================================================
raw_data_path = r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed\insurance_data_cleaned.csv"  # Update this
output_dir = r"D:\Personal\KAIM-10 Academy\Week 3\Project Work\Insurance Analytics_Predictive Modeling-Week 3\data\processed"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Check if file exists
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"❌ Raw CSV file not found at {raw_data_path}")

# ===========================================================
# 1. Load Data
# ===========================================================
df = pd.read_csv(raw_data_path, low_memory=False)

# Normalize column names
df.columns = df.columns.str.lower().str.strip()

print("✅ Cleaned Data Loaded Successfully!", df.shape)

# ===========================================================
# 2. Define Targets
# ===========================================================
# Classification target: did a claim occur?
df['claim_occurred'] = (df['totalclaims'] > 0).astype(int)
target_cls = 'claim_occurred'

# Regression target: claim amount
target_reg = 'totalclaims'

print("✅ Targets defined successfully")

# ===========================================================
# 3. Fix Mixed-Type Columns
# ===========================================================
# Convert numeric-looking columns safely
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

# Identify numeric and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

print("🔎 Numerical columns:", len(num_cols))
print("🔎 Categorical columns:", len(cat_cols))

# ===========================================================
# 4. Drop Empty Numeric Columns
# ===========================================================
empty_numeric = [col for col in num_cols if df[col].isna().sum() == len(df)]
if empty_numeric:
    print("⚠️ Dropping empty numeric columns:", empty_numeric)
    df.drop(columns=empty_numeric, inplace=True)
    num_cols = [c for c in num_cols if c not in empty_numeric]

# ===========================================================
# 5. Handle Missing Values
# ===========================================================
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

df[num_cols] = num_imputer.fit_transform(df[num_cols])
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("✅ Missing values handled")

# ===========================================================
# 6. Feature Engineering
# ===========================================================
# Car Age
if "registrationyear" in df.columns:
    df["car_age"] = 2024 - df["registrationyear"]

# Engine size category
if "cubiccapacity" in df.columns:
    df["engine_size_cat"] = pd.cut(
        df["cubiccapacity"],
        bins=[0, 1200, 1600, 2000, 3000, np.inf],
        labels=["Small", "Medium", "Standard", "Large", "XL"]
    )

# mmcode prefix
if "mmcode" in df.columns:
    df["mmcode_prefix"] = df["mmcode"].astype(str).str[:2]

print("✅ Feature Engineering Completed")

# ===========================================================
# 7. Encode Categorical Columns
# ===========================================================
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Label encode high-cardinality, one-hot encode low-cardinality
label_cols = [col for col in cat_cols if df[col].nunique() > 30]
ohe_cols = [col for col in cat_cols if df[col].nunique() <= 30]

# Label encode
le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# One-hot encode
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print("✅ Encoding Completed")

# ===========================================================
# 8. Scale Numeric Columns
# ===========================================================
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

print("✅ Scaling Completed")

# ===========================================================
# 9. Train-Test Split
# ===========================================================
# Classification
X_cls = df.drop(columns=[target_cls])
y_cls = df[target_cls]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42, stratify=y_cls
)

# Regression
X_reg = df.drop(columns=[target_reg])
y_reg = df[target_reg]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

print("✅ Train-Test Split Completed")
print("Classification Train:", X_train_c.shape)
print("Regression Train:", X_train_r.shape)

# ===========================================================
# 10. Save Processed Files
# ===========================================================
X_train_c.to_csv(os.path.join(output_dir, "X_train_classification.csv"), index=False)
X_test_c.to_csv(os.path.join(output_dir, "X_test_classification.csv"), index=False)
y_train_c.to_csv(os.path.join(output_dir, "y_train_classification.csv"), index=False)
y_test_c.to_csv(os.path.join(output_dir, "y_test_classification.csv"), index=False)

X_train_r.to_csv(os.path.join(output_dir, "X_train_regression.csv"), index=False)
X_test_r.to_csv(os.path.join(output_dir, "X_test_regression.csv"), index=False)
y_train_r.to_csv(os.path.join(output_dir, "y_train_regression.csv"), index=False)
y_test_r.to_csv(os.path.join(output_dir, "y_test_regression.csv"), index=False)

df.to_csv(os.path.join(output_dir, "model_ready_full.csv"), index=False)

print("🎉 All processed files saved successfully!")
